In [2]:
import sys
from pathlib import Path
from uncertainties import ufloat
import country_converter as coco
import matplotlib.pyplot as plt
import re
import pandas as pd
import math
import numpy as np
from uncertainties import ufloat, nominal_value, std_dev
from uncertainties.core import CallableStdDev

# ---------------------------- Run from Repo Root ----------------------------
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

# ---------------------------- File Paths ----------------------------
country_list_csv = BASE_DIR / "data" / "STANDARD_COUNTRY_LIST.csv"
CR_Box_Countries = BASE_DIR / "data" / "CR_Box_Countries_MS.csv"
ecw_country = BASE_DIR / "results" / "ECWbyCountry.csv"
ecw_region = BASE_DIR / "results" / "ECWbyRegion.csv"
Baghouse_Airflow = BASE_DIR /'data'/"BaghouseAirflow.csv"
SRC_DIR = BASE_DIR / "src"
sys.path.append(str(SRC_DIR)) 

from country_pkg import Country

In [8]:
# ------------------------ Functions ----------------------------

def generate_countries_from_multiple_csvs(
    country_csv_path,
    cr_box_csv_path=None,
    ecw_csv_path=None,
    baghouse_csv_path=None
):
    # ---------------- Main country CSV ----------------
    df = pd.read_csv(country_csv_path, encoding='cp1252')
    required_cols = ['ISO-3', 'Country Name']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must have a column named '{col}'")
    
    # ---------------- CR Box CSV ----------------
    cr_box_df = None
    if cr_box_csv_path:
        cr_box_df = pd.read_csv(cr_box_csv_path, encoding='cp1252')
        if "Country" not in cr_box_df.columns:
            raise ValueError("CR Box CSV must have a 'Country' column")
        cr_box_df["Country"] = cr_box_df["Country"].apply(Country._cc.convert, to="name_short")
    
    # ---------------- ECW CSV ----------------
    ecw_df = None
    if ecw_csv_path:
        ecw_df = pd.read_csv(ecw_csv_path, encoding='cp1252')
        required_ecw_cols = ['Country Name', 'Country Code']
        for col in required_ecw_cols:
            if col not in ecw_df.columns:
                raise ValueError(f"ECW CSV must have a column named '{col}'")
    
    # ---------------- Baghouse Airflow CSV ----------------
    baghouse_df = None
    if baghouse_csv_path:
        baghouse_df = pd.read_csv(baghouse_csv_path, encoding='cp1252')
        required_baghouse_cols = ['Country', 'Operating MW']
        for col in required_baghouse_cols:
            if col not in baghouse_df.columns:
                raise ValueError(f"Baghouse CSV must have a column named '{col}'")
        # Standardize country names
        baghouse_df["Country"] = baghouse_df["Country"].apply(Country._cc.convert, to="name_short")
    
    countries = {}
    
    for _, row in df.iterrows():
        iso_code = row['ISO-3']
        country_name = row['Country Name']
        
        # Create country object
        c = Country(name=iso_code)
        c.properties['ISO-3'] = iso_code
        
        # ---------------- Merge CR Box properties ----------------
        if cr_box_df is not None:
            standardized_name = Country._cc.convert(country_name, to="name_short")
            cr_row = cr_box_df[cr_box_df["Country"] == standardized_name]
            if not cr_row.empty:
                for col in cr_row.columns:
                    if col != "Country":
                        c.properties[col] = cr_row.iloc[0][col]
            else:
                for col in cr_box_df.columns:
                    if col != "Country":
                        c.properties[col] = 0
        
        # ---------------- Merge ECW properties ----------------
        if ecw_df is not None:
            ecw_row = ecw_df[ecw_df["Country Code"] == iso_code]
            if not ecw_row.empty:
                for col in ecw_row.columns:
                    if col not in ["Country Code", "Country Name"]:
                        c.properties[col] = ecw_row.iloc[0][col]
        
        # ---------------- Merge Baghouse properties ----------------
        if baghouse_df is not None:
            standardized_name = Country._cc.convert(country_name, to="name_short")
            baghouse_row = baghouse_df[baghouse_df["Country"] == standardized_name]
            if not baghouse_row.empty:
                c.properties["Baghouse Operating MW"] = baghouse_row.iloc[0]["Operating MW"]
            else:
                c.properties["Baghouse Operating MW"] = 0
        
        countries[iso_code] = c
    
    return countries

def manufacturing_delay_function(M):
    if M > 90: return 1
    elif M > 85: return 2
    elif M > 80: return 3
    elif M > 75: return 4
    elif M > 70: return 5
    elif M > 65: return 6
    elif M > 60: return 7
    elif M > 55.5: return 8
    else: return 0

def scale_up_MAIN(country,t):
    i = 1
    data_points = [0]
    while i <= t:
        prev = data_points[-1]
        next = prev

        # --- Initial Increases ---
        if i == country.properties["Repurposing and Initial Stock Delay"]:
            next += country.properties["CADR: CR Box Repurposing"]
            next += country.properties["CADR: CR Box Initial Stock"]
        if i == country.properties['Coalbaghouse Delay']:
            next += country.properties['CADR: Coal Baghouse']
        
        # --- Ongoing Increases ---
        if country.properties["Big_6"]:
            delay_til_100 = CR_Box_70_til_100_Delay+country.properties["CR Box Manufacturing Distribution Delay"]
            if i < country.properties["CR Box Manufacturing Distribution Delay"]:
                next += 0
            if i < delay_til_100 and  i >= country.properties["CR Box Manufacturing Distribution Delay"]:
                next += (0.7+(0.05*(i-1)))*country.properties["CADR: CR Box Weekly Production"]
            if i >= delay_til_100:
                next += country.properties["CADR: CR Box Weekly Production"]
        else:
            if i >= country.properties["CR Box Manufacturing Distribution Delay"]:
                next += country.properties["CADR: CR Box Weekly Production"]

        data_points.append(next)
        i=i+1
    return data_points

def scale_up_CR_MAN(country,t):
    i = 1
    data_points = [0]
    while i <= t:
        prev = data_points[-1]
        next = prev        
        if country.properties["Big_6"]:
            delay_til_100 = CR_Box_70_til_100_Delay+country.properties["CR Box Manufacturing Distribution Delay"]
            if i < country.properties["CR Box Manufacturing Distribution Delay"]:
                next += 0
            if i < delay_til_100 and  i >= country.properties["CR Box Manufacturing Distribution Delay"]:
                next += (0.7+(0.05*(i-1)))*country.properties["CADR: CR Box Weekly Production"]
            if i >= delay_til_100:
                next += country.properties["CADR: CR Box Weekly Production"]
        else:
            if i >= country.properties["CR Box Manufacturing Distribution Delay"]:
                next += country.properties["CADR: CR Box Weekly Production"]
        data_points.append(next)
        i=i+1
    return data_points

def scale_up_CR_REPUR(country,t):
    i = 1
    data_points = [0]
    while i <= t:
        prev = data_points[-1]
        next = prev
        if i == country.properties["Repurposing and Initial Stock Delay"]:
            next += country.properties["CADR: CR Box Repurposing"]
        data_points.append(next)
        i=i+1
    return data_points

def scale_up_CR_STOCK(country,t):
    i = 1
    data_points = [0]
    while i <= t:
        prev = data_points[-1]
        next = prev
        if i == country.properties["Repurposing and Initial Stock Delay"]:
            next += country.properties["CADR: CR Box Initial Stock"]
        data_points.append(next)
        i=i+1
    return data_points

def scale_up_COALBAG(country,t):
    i = 1
    data_points = [0]
    while i <= t:
        prev = data_points[-1]
        next = prev
        if i == country.properties['Coalbaghouse Delay']:
            next += country.properties['CADR: Coal Baghouse']
        data_points.append(next)
        i=i+1
    return data_points


In [4]:
## ---------------------------- Varaibles ---------------------------- 

CR_Box_CADR_LS = 126.13
Filter_Life_Span = ufloat((2-1)/2 , (2-1)/4)
Scale_Up_Factor = 1/0.7
Initial_Stock_in_weeks = ufloat(6,0.5)
Factory_minimum_production = 50000
Coalbaghouse_efficency = ufloat((0.8+0.5)/2, (0.8-0.5)/4)
Coalbaghouse_gradient = ufloat(1717, 419.3/2)
Coalbaghouse_offset = 33807
Coalbaghouse_Utilisation = ufloat(0.3,0.1)
Repurposing_and_Initial_Stock_Delay = 2
Coalbaghouse_Delay = 4
CR_Box_70_til_100_Delay = 6

UNRegion_list = ['Australia and New Zealand',
 'Caribbean',
 'Central America',
 'Central Asia',
 'Eastern Africa',
 'Eastern Asia',
 'Eastern Europe',
 'Melanesia',
 'Micronesia',
 'Middle Africa',
 'Northern Africa',
 'Northern America',
 'Northern Europe',
 'Polynesia',
 'South America',
 'South-eastern Asia',
 'Southern Africa',
 'Southern Asia',
 'Southern Europe',
 'Western Africa',
 'Western Asia',
 'Western Europe']


In [5]:
## ---------------------------- CR Box Reference ---------------------------- 

Ind_Market_Rev_Per_MERV = {
    '17-20' : 2208.7e6,
    '5-8'   : 563.4e6,
    '9-12'  : 1271.8e6,
    '1-4'   : 171.7e6,
    '13-16' : 1878.1e6}

Tot_Ind_Air_Filter = Ind_Market_Rev_Per_MERV['17-20']+Ind_Market_Rev_Per_MERV['5-8']+Ind_Market_Rev_Per_MERV['1-4']+Ind_Market_Rev_Per_MERV['9-12']+Ind_Market_Rev_Per_MERV['13-16']

Tot_Air_Filter = 20.8303e9

All_Market_Rev_Per_MERV = {
    '17-20' : Ind_Market_Rev_Per_MERV['17-20']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '5-8'   : Ind_Market_Rev_Per_MERV['5-8']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '9-12'  : Ind_Market_Rev_Per_MERV['9-12']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '1-4'   : Ind_Market_Rev_Per_MERV['1-4']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '13-16' : Ind_Market_Rev_Per_MERV['13-16']*Tot_Air_Filter/Tot_Ind_Air_Filter}

Price_Per_Filter = {
    '1-4'   : ufloat(1031.59,   107.26/2),
    '5-8'   : ufloat(1133.85,   447.48/2),
    '9-12'  : ufloat(1302.51,   554.53/2),
    '13-16' : ufloat(1951.25,   593.63/2),
    '17-20' : ufloat(22925.29,  3740.81/2)}

Volume_to_Sale = 0.508*0.508*0.0254

Sales = {
    '1-4'   : All_Market_Rev_Per_MERV['1-4']/(Price_Per_Filter['1-4']*Volume_to_Sale),
    '5-8'   : All_Market_Rev_Per_MERV['5-8']/(Price_Per_Filter['5-8']*Volume_to_Sale),
    '9-12'  : All_Market_Rev_Per_MERV['9-12']/(Price_Per_Filter['9-12']*Volume_to_Sale),
    '13-16' : All_Market_Rev_Per_MERV['13-16']/(Price_Per_Filter['13-16']*Volume_to_Sale),
    '17-20' : All_Market_Rev_Per_MERV['17-20']/(Price_Per_Filter['17-20']*Volume_to_Sale)}

Panel_Filter = ufloat(0.35,     0.35*0.1/2)

Usable_Filters = (Sales['13-16']+Sales['17-20']) * Panel_Filter * Scale_Up_Factor

Repurposeable_Filters = (Sales['13-16']+Sales['17-20']) * Panel_Filter * Filter_Life_Span


(1.0+/-0.5)e+08


In [23]:
## ---------------------------- Countries ---------------------------- 

if "countries_dict" not in globals():
    countries_dict = generate_countries_from_multiple_csvs(country_list_csv, CR_Box_Countries, ecw_country, Baghouse_Airflow)

# ---------- CR Box Scale Factor ----------
sum_scale = 0
for country in countries_dict.values():
    msa = country.properties["MSA"]
    mva = country.properties["MVA"]
    if msa == 1:
        country.properties["Big_6"] = True
        sum_scale += mva
    else:
        country.properties["Big_6"] = False
scale = Usable_Filters/sum_scale

# ---------- Country Calculations ----------
for country in countries_dict.values():
    # ------ CR Box Manufacturing ------
    msa = country.properties["MSA"]
    mva = country.properties["MVA"]
    x = scale * msa * mva if mva > 55.5 else ufloat(0,0)
    x = ufloat(0, 0) if x.nominal_value < Factory_minimum_production else x / 4
    country.properties["CR Box Annual Production"] = ufloat(math.floor(x.nominal_value), x.std_dev)
    country.properties["CR Box Weekly Production"] = country.properties["CR Box Annual Production"] / 52
    country.properties["CADR: CR Box Annual Production"] = country.properties["CR Box Annual Production"] * CR_Box_CADR_LS
    country.properties["CADR: CR Box Weekly Production"] = country.properties["CR Box Weekly Production"] * CR_Box_CADR_LS

    # ------ CR Box Initial Stock ------
    if country.properties["Big_6"] == True:
        country.properties["CR Box Initial Stock"] = country.properties["CR Box Weekly Production"]*0.7 * Initial_Stock_in_weeks
        country.properties["CADR: CR Box Initial Stock"] = country.properties["CR Box Initial Stock"]*CR_Box_CADR_LS
    else: 
        country.properties["CR Box Initial Stock"] = ufloat(0,0)
        country.properties["CADR: CR Box Initial Stock"] = ufloat(0,0)

    # --- Repurposing of CR Boxes---
    global_MVA=0
    for c in countries_dict.values():
        global_MVA = global_MVA+ c.properties["MVA"]
    country.properties['Ave ECW %'] = ufloat((country.properties["%ECW ILO"] + country.properties["%ECW Poll"] )/2,(country.properties["%ECW ILO"] - country.properties["%ECW Poll"] )/4)
    rel_mva = country.properties["MVA"]/global_MVA
    country.properties['CR Box Repurposing'] = Repurposeable_Filters * rel_mva * country.properties['Ave ECW %']
    country.properties['CADR: CR Box Repurposing'] = country.properties['CR Box Repurposing']*CR_Box_CADR_LS

    # ------ Coalbaghouse Calculations ------
    country.properties['CADR: Coal Baghouse'] = Coalbaghouse_efficency*Coalbaghouse_Utilisation*(Coalbaghouse_gradient * country.properties['Baghouse Operating MW'] + Coalbaghouse_offset)
    
    # ------ Delay Function ------
    # --- Manufacturing Delay ---
    m = country.properties["MFS"]
    country.properties['CR Box Manufacturing Distribution Delay'] = manufacturing_delay_function(m)
    # --- Repurposing and Initial Stock Delay ---
    country.properties['Repurposing and Initial Stock Delay'] = Repurposing_and_Initial_Stock_Delay
    # --- Repurposing and Initial Stock Delay ---
    country.properties['Coalbaghouse Delay'] = Coalbaghouse_Delay
    # country.summary()


--- Germany ---
ISO-3: DEU
MFS: 91.0
MVA: 838894000000.0
MSA: 1.0
Labour Force (2024): 43800000.0
Region: Western Europe
%ECW ILO: 0.315472660092244
%ECW Poll: 0.1560926482683352
ECW ILO: 13817702.51204029
ECW Poll: 6836857.994153083
Baghouse Operating MW: 28970.1
Big_6: True
CR Box Annual Production: (5.0+/-0.7)e+06
CR Box Weekly Production: (9.6+/-1.4)e+04
CADR: CR Box Annual Production: (6.3+/-0.9)e+08
CADR: CR Box Weekly Production: (1.21+/-0.18)e+07
CR Box Initial Stock: (4.0+/-0.7)e+05
CADR: CR Box Initial Stock: (5.1+/-0.9)e+07
Ave ECW %: 0.24+/-0.04
CR Box Repurposing: (1.1+/-0.6)e+06
CADR: CR Box Repurposing: (1.4+/-0.7)e+08
CADR: Coal Baghouse: (1.0+/-0.4)e+07
CR Box Manufacturing Distribution Delay: 1
Repurposing and Initial Stock Delay: 2
Coalbaghouse Delay: 4


In [37]:
## ---------------------------- Scale Up ---------------------------- 

main_scale_up_path = BASE_DIR / "results" / "Scale_up_output_MS.csv"

cr_man_scale_up_path = BASE_DIR / "results" / "Scale_up_CR_MAN_MS.csv"
cr_repur_scale_up_path = BASE_DIR / "results" / "Scale_up_CR_REPUR_MS.csv"
cr_stock_scale_up_path = BASE_DIR / "results" / "Scale_up_CR_STOCK.csv"
coalbag_scale_up_path = BASE_DIR / "results" / "Scale_up_COALBAG_MS.csv"

Scale_Up_Time_Period_In_Weeks = 52

## ------- Country Scale Up Dictionary -------
scale_up_data_country_MAIN ={}

scale_up_data_country_CR_MAN ={}
scale_up_data_country_CR_REPUR ={}
scale_up_data_country_CR_STOCK ={}
scale_up_data_country_COALBAG ={}

## ------- Region Scale Up Dictionary -------
scale_up_data_region_MAIN = {}

scale_up_data_region_CR_MAN = {}
scale_up_data_region_CR_REPUR = {}
scale_up_data_region_CR_STOCK = {}
scale_up_data_region_COALBAG = {}

for region in UNRegion_list:
    scale_up_data_region_MAIN[region] = [0]*(Scale_Up_Time_Period_In_Weeks+3)

    scale_up_data_region_CR_MAN[region] = [0]*(Scale_Up_Time_Period_In_Weeks+3)
    scale_up_data_region_CR_REPUR[region] = [0]*(Scale_Up_Time_Period_In_Weeks+3)
    scale_up_data_region_CR_STOCK[region] = [0]*(Scale_Up_Time_Period_In_Weeks+3)
    scale_up_data_region_COALBAG[region] = [0]*(Scale_Up_Time_Period_In_Weeks+3)

## ------- Scale Up Calculations -------
for country in countries_dict.values():
    MAIN_data_points = scale_up_MAIN(country, Scale_Up_Time_Period_In_Weeks)
    
    CR_MAN_data_points = scale_up_CR_MAN(country, Scale_Up_Time_Period_In_Weeks)
    CR_REPUR_data_points = scale_up_CR_REPUR(country, Scale_Up_Time_Period_In_Weeks)
    CR_STOCK_data_points = scale_up_CR_STOCK(country, Scale_Up_Time_Period_In_Weeks)
    COALBAG_data_points = scale_up_COALBAG(country, Scale_Up_Time_Period_In_Weeks)

    ecw_list = [
        0 if np.isnan(country.properties['ECW Poll']) else int(country.properties['ECW Poll']),
        0 if np.isnan(country.properties['ECW ILO']) else int(country.properties['ECW ILO'])
    ]    
    scale_up_data_country_MAIN[country.name] = ecw_list + MAIN_data_points

    scale_up_data_country_CR_MAN[country.name] = ecw_list + CR_MAN_data_points
    scale_up_data_country_CR_REPUR[country.name] = ecw_list + CR_REPUR_data_points
    scale_up_data_country_CR_STOCK[country.name] = ecw_list + CR_STOCK_data_points
    scale_up_data_country_COALBAG[country.name] = ecw_list + COALBAG_data_points

    for region in UNRegion_list: 
        if country.properties['Region'] == region:
            scale_up_data_region_MAIN[region]=list(np.add(scale_up_data_country_MAIN[country.name],scale_up_data_region_MAIN[region]))

            scale_up_data_region_CR_MAN[region]=list(np.add(scale_up_data_country_CR_MAN[country.name],scale_up_data_region_CR_MAN[region]))
            scale_up_data_region_CR_REPUR[region]=list(np.add(scale_up_data_country_CR_REPUR[country.name],scale_up_data_region_CR_REPUR[region]))
            scale_up_data_region_CR_STOCK[region]=list(np.add(scale_up_data_country_CR_STOCK[country.name],scale_up_data_region_CR_STOCK[region]))
            scale_up_data_region_COALBAG[region]=list(np.add(scale_up_data_country_COALBAG[country.name],scale_up_data_region_COALBAG[region]))


## ------- Global Calculation -------
global_data_points_MAIN = [0]*(Scale_Up_Time_Period_In_Weeks+3)

global_data_points_CR_MAN = [0]*(Scale_Up_Time_Period_In_Weeks+3)
global_data_points_CR_REPUR = [0]*(Scale_Up_Time_Period_In_Weeks+3)
global_data_points_CR_STOCK = [0]*(Scale_Up_Time_Period_In_Weeks+3)
global_data_points_COALBAG = [0]*(Scale_Up_Time_Period_In_Weeks+3)

for country in scale_up_data_country_MAIN:
    global_data_points_MAIN = list(np.add(global_data_points_MAIN, scale_up_data_country_MAIN[country]))

    global_data_points_CR_MAN = list(np.add(global_data_points_CR_MAN, scale_up_data_country_CR_MAN[country]))
    global_data_points_CR_REPUR = list(np.add(global_data_points_CR_REPUR, scale_up_data_country_CR_REPUR[country]))
    global_data_points_CR_STOCK = list(np.add(global_data_points_CR_STOCK, scale_up_data_country_CR_STOCK[country]))
    global_data_points_COALBAG = list(np.add(global_data_points_COALBAG, scale_up_data_country_COALBAG[country]))

scale_up_data_region_MAIN['Global']=global_data_points_MAIN

scale_up_data_region_CR_MAN['Global']=global_data_points_CR_MAN
scale_up_data_region_CR_REPUR['Global']=global_data_points_CR_REPUR
scale_up_data_region_CR_STOCK['Global']=global_data_points_CR_STOCK
scale_up_data_region_COALBAG['Global']=global_data_points_COALBAG


## ------- Exporting -------
Scale_Up_df_MAIN = pd.DataFrame(scale_up_data_country_MAIN | scale_up_data_region_MAIN).T
Scale_Up_df_MAIN.to_csv(main_scale_up_path)
Scale_Up_df_MAIN.to_pickle(main_scale_up_path.with_suffix(".pkl"))

Scale_Up_df_CR_MAN = pd.DataFrame(scale_up_data_country_CR_MAN | scale_up_data_region_CR_MAN).T
Scale_Up_df_CR_MAN.to_csv(cr_man_scale_up_path)
Scale_Up_df_CR_MAN.to_pickle(cr_man_scale_up_path.with_suffix(".pkl"))
Scale_Up_df_CR_REPUR = pd.DataFrame(scale_up_data_country_CR_REPUR | scale_up_data_region_CR_REPUR).T
Scale_Up_df_CR_REPUR.to_csv(cr_repur_scale_up_path)
Scale_Up_df_CR_REPUR.to_pickle(cr_repur_scale_up_path.with_suffix(".pkl"))
Scale_Up_df_CR_STOCK = pd.DataFrame(scale_up_data_country_CR_STOCK | scale_up_data_region_CR_STOCK).T
Scale_Up_df_CR_STOCK.to_csv(cr_stock_scale_up_path)
Scale_Up_df_CR_STOCK.to_pickle(cr_stock_scale_up_path.with_suffix(".pkl"))
Scale_Up_df_COALBAG = pd.DataFrame(scale_up_data_country_COALBAG | scale_up_data_region_COALBAG).T
Scale_Up_df_COALBAG.to_csv(coalbag_scale_up_path)
Scale_Up_df_COALBAG.to_pickle(coalbag_scale_up_path.with_suffix(".pkl"))
